# Day 16 – RAG Pipeline

**RAG = Retrieve → Augment → Generate**

Instead of relying on the LLM's training knowledge, RAG:
1. **Retrieves** relevant chunks from your documents
2. **Augments** the user's question with those chunks
3. **Generates** an answer grounded in your documents

This notebook builds a complete RAG system and compares it against a plain LLM.

## RAG Architecture

```
INDEXING (done once):
  Documents → Split into chunks → Embed each chunk → Store in ChromaDB

QUERYING (done per question):
  Question → Embed question → Find similar chunks → Send chunks + question to LLM → Answer
```

In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

load_dotenv(dotenv_path="../../../.env")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
print("LLM and embeddings ready.")

LLM and embeddings ready.


---
## Step 1: Load Documents

In [ ]:
loader = DirectoryLoader("documents/", glob="*.txt", loader_cls=TextLoader)
documents = loader.load()

print(f"Loaded {len(documents)} documents:")
for doc in documents:
    print(f"  - {doc.metadata['source']} ({len(doc.page_content)} chars)")

Loaded 3 documents:
  - documents/ml_concepts.txt (2376 chars)
  - documents/python_tips.txt (3044 chars)
  - documents/india_space.txt (2483 chars)


---
## Step 2: Split into Chunks

Documents are too long to embed as a whole. We split them into smaller overlapping chunks.
- `chunk_size`: max characters per chunk
- `chunk_overlap`: characters shared between adjacent chunks (preserves context at boundaries)

In [12]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

print(f"Split {len(documents)} documents into {len(chunks)} chunks.")
print(f"\nExample chunk:")
print(f"Source : {chunks[0].metadata['source']}")
print(f"Content: {chunks[0].page_content[:300]}")

Split 3 documents into 26 chunks.

Example chunk:
Source : documents/ml_concepts.txt
Content: Machine Learning Concepts

Supervised Learning
Supervised learning is a type of machine learning where the model is trained on labeled data. Each training example has an input and a known correct output. The model learns to map inputs to outputs. Common examples include linear regression for predict


---
## Step 3: Embed Chunks and Store in ChromaDB

In [4]:
# This calls the OpenAI embeddings API for each chunk, then stores in ChromaDB
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_rag_db"
)

print(f"Stored {vectorstore._collection.count()} chunks in ChromaDB.")
print("Persisted to ./chroma_rag_db")

Stored 26 chunks in ChromaDB.
Persisted to ./chroma_rag_db


---
## Step 4: Semantic Search – Retrieve Relevant Chunks

In [5]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

query = "What is overfitting and how do I fix it?"
relevant_docs = retriever.invoke(query)

print(f"Query: {query}")
print(f"\nTop {len(relevant_docs)} retrieved chunks:")
for i, doc in enumerate(relevant_docs, 1):
    source = os.path.basename(doc.metadata['source'])
    print(f"\n[Chunk {i} from {source}]")
    print(doc.page_content)

Query: What is overfitting and how do I fix it?

Top 3 retrieved chunks:

[Chunk 1 from ml_concepts.txt]
Overfitting
Overfitting happens when a model learns the training data too well, including its noise. The model performs well on training data but poorly on new unseen data. Solutions include adding more training data, using dropout, applying regularization (L1/L2), and simplifying the model.

[Chunk 2 from ml_concepts.txt]
Underfitting
Underfitting occurs when a model is too simple to capture the underlying pattern in the data. Both training and test accuracy are low. Solutions include using a more complex model, adding more features, and training for more epochs.

[Chunk 3 from ml_concepts.txt]
Cross Validation
Cross validation is a technique to evaluate model performance on unseen data. In k-fold cross validation, the dataset is split into k equal parts. The model is trained on k-1 parts and tested on the remaining part. This is repeated k times and results are averaged.


---
## Step 5: Full RAG Chain (Retrieve + Generate)

`RetrievalQA` automatically retrieves relevant chunks and sends them to the LLM as context.

In [6]:
# Custom prompt that injects retrieved context
rag_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=(
        "Answer the question using only the context below. "
        "If the answer is not in the context, say 'I don't have information about that.'\n\n"
        "Context:\n{context}\n\n"
        "Question: {question}\n"
        "Answer:"
    )
)

rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",        # 'stuff' = put all chunks into one prompt
    retriever=retriever,
    chain_type_kwargs={"prompt": rag_prompt},
    return_source_documents=True
)

def ask(question):
    result = rag_chain.invoke({"query": question})
    print(f"Q: {question}")
    print(f"A: {result['result']}")
    sources = {os.path.basename(d.metadata['source']) for d in result['source_documents']}
    print(f"Sources: {', '.join(sources)}\n")

ask("What is overfitting and how can I prevent it?")

Q: What is overfitting and how can I prevent it?
A: Overfitting happens when a model learns the training data too well, including its noise, resulting in good performance on training data but poor performance on new unseen data. To prevent overfitting, you can add more training data, use dropout, apply regularization (L1/L2), and simplify the model.
Sources: ml_concepts.txt



In [7]:
ask("When did Chandrayaan-3 land on the Moon?")
ask("What is a context manager in Python?")
ask("How much did the Mangalyaan mission cost?")

Q: When did Chandrayaan-3 land on the Moon?
A: Chandrayaan-3 landed on the Moon on August 23, 2023.
Sources: india_space.txt

Q: What is a context manager in Python?
A: A context manager in Python handles setup and teardown logic automatically using the with statement. It ensures that resources, like files, are properly managed, such as being closed even if an exception occurs.
Sources: python_tips.txt

Q: How much did the Mangalyaan mission cost?
A: The Mangalyaan mission cost about Rs 450 crore.
Sources: india_space.txt



In [8]:
# Question outside the documents — RAG should say it doesn't know
ask("What is the capital of France?")

Q: What is the capital of France?
A: I don't have information about that.
Sources: python_tips.txt



---
## Step 6: Direct LLM vs RAG – Comparison

The LLM on its own answers from training data — which may be wrong or generic.
RAG grounds the answer in your specific documents.

In [9]:
from langchain_core.output_parsers import StrOutputParser
from langchain.prompts import ChatPromptTemplate

direct_prompt = ChatPromptTemplate.from_messages([
    ("human", "{question}")
])
direct_chain = direct_prompt | llm | StrOutputParser()

test_questions = [
    "What is the exact cost of the Chandrayaan-3 mission?",
    "What does the PSLV-C37 mission hold the record for?",
    "What are the solutions for overfitting according to the ML concepts document?",
]

for q in test_questions:
    print("=" * 60)
    print(f"QUESTION: {q}")
    print("\n--- Direct LLM ---")
    print(direct_chain.invoke({"question": q}))
    print("\n--- RAG ---")
    result = rag_chain.invoke({"query": q})
    print(result["result"])
    print()

QUESTION: What is the exact cost of the Chandrayaan-3 mission?

--- Direct LLM ---
The Chandrayaan-3 mission, launched by the Indian Space Research Organisation (ISRO) in July 2023, had an estimated cost of around ₹615 crore (approximately $75 million USD). This mission aimed to demonstrate India's capability for a soft landing on the Moon and included a lander and a rover. For the most accurate and up-to-date information, it's always best to refer to official ISRO announcements or reports.

--- RAG ---
The total mission cost of Chandrayaan-3 was approximately Rs 615 crore.

QUESTION: What does the PSLV-C37 mission hold the record for?

--- Direct LLM ---
The PSLV-C37 mission, launched by the Indian Space Research Organisation (ISRO) on February 15, 2017, holds the record for launching the highest number of satellites in a single mission. During this mission, PSLV-C37 successfully deployed a total of 104 satellites into orbit, surpassing the previous record of 37 satellites launched by

---
## Step 7: Reload Persisted Vector Store

Once built, you don't need to re-embed. Load the saved ChromaDB directly.

In [10]:
# Load the existing ChromaDB without re-embedding
loaded_store = Chroma(
    persist_directory="./chroma_rag_db",
    embedding_function=embeddings
)

print(f"Reloaded {loaded_store._collection.count()} chunks from disk.")

# Use it exactly the same way
loaded_retriever = loaded_store.as_retriever(search_kwargs={"k": 3})
docs = loaded_retriever.invoke("how does gradient descent work")
print(f"Retrieved {len(docs)} chunks for 'how does gradient descent work'")

Reloaded 26 chunks from disk.
Retrieved 3 chunks for 'how does gradient descent work'
